##### Research Insight – Multimodal RAG PDF QA

This project is a small research assistant that can read a PDF and answer questions from it.
It works on text, tables, and images inside the document using a multimodal RAG pipeline.
The main goal was to build something practical that can understand research papers better than normal text-only systems.

In [ ]:
#!pip install -U google-generativeai langchain langchain-google-genai python-dotenv

In [ ]:
#!pip install -U google-ai-generativelanguage

In [ ]:
#!pip install -U pillow

In [ ]:
from dotenv import load_dotenv
load_dotenv()

tests w/ langchain:
1. gemini-pro
2. gemini-pro-vision

In [ ]:
from langchain_core.messages import HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
#import PIL.Image
#from PIL import Image
import urllib.request
import requests
import google.generativeai as genai

In [ ]:
# supported models
for m in genai.list_models():
  if 'generateContent' in m.supported_generation_methods:
    print(m.name)

In [ ]:
# language model from langchain
llm = ChatGoogleGenerativeAI(model="gemini-pro", temperature =0.7)
llm.invoke("what is a Large Language Model").content

In [ ]:
llm

In [ ]:
# open with PIL
image_url ="https://cioviews.com/wp-content/uploads/2020/12/2-2.jpg"
urllib.request.urlretrieve(image_url, "test.png") 
img = Image.open("test.png")
img

In [ ]:
from IPython.display import display
from IPython.display import Markdown
import textwrap

def to_markdown(text):
  text = text.replace('•', '  *')
  return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

In [ ]:
# alternative
from IPython.display import Image
from IPython.core.display import HTML
img = Image('test.jpg')
#img = Image(url="https://cioviews.com/wp-content/uploads/2020/12/2-2.jpg") # does not work with vison model

In [ ]:
img

In [ ]:
# Multimodal model
#import PIL.Image
#img = PIL.Image.open("test.png")
model = genai.GenerativeModel('gemini-pro-vision')
response = model.generate_content(img)

In [ ]:
to_markdown(response.text)

In [ ]:
# Multimodal model w/ langchain
from PIL import Image
img = Image.open("test.png")

llm = ChatGoogleGenerativeAI(model="gemini-pro-vision", temperature=0.2)
# example
message = HumanMessage(
    content=[
        {
            "type": "text",
            "text": "What's in this image?",
        },  # You can optionally provide text parts
        {"type": "image_url", "image_url": { "url": "https://cioviews.com/wp-content/uploads/2020/12/2-2.jpg"} }, # img  
    ]
)

In [ ]:
to_markdown(llm.invoke([message]).content)

## Data Ingestion (pdf)

In [ ]:
#!pip install "unstructured[all-docs]" chromadb pydantic lxml tiktoken

In [ ]:
#!pip install pytesseract

**Extract tables and images using ## unstructured**

install poppler and tesseract

In [ ]:
from unstructured.partition.pdf import partition_pdf
import pytesseract

In [ ]:
pytesseract.pytesseract.tesseract_cmd = r'D:\Manthan_Shenoy\Manthan_new\projects\projects-NLP\multimodal-rag-gemini\Tesseract-OCR\tesseract.exe'

In [ ]:
image_path = "./"
pdf_elements = partition_pdf(
    "test.pdf",
    chunking_strategy="by_title",
    extract_images_in_pdf=True,
    infer_table_structure=True,
    max_characters=3000,
    new_after_n_chars=2800,
    combine_text_under_n_chars=2000,
    image_output_dir_path=image_path
    )

tables and texts into different groups

In [ ]:
# Categorize elements by type
def categorize_elements(raw_pdf_elements):
    text_elements = []
    table_elements = []
    for element in raw_pdf_elements:
        if 'CompositeElement' in str(type(element)):
            text_elements.append(str(element))
        elif 'Table' in str(type(element)):
            table_elements.append(str(element))
    return text_elements, table_elements


In [ ]:
# extract tables and texts
texts, tables = categorize_elements(pdf_elements)

# length of text elem
print(len(texts))

# length of table elem
print(len(tables))

In [ ]:
"""
import os
import base64
image_path = "./figures"
image_elements = []

# Function to encode images
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

for image_file in os.listdir(image_path):
    if image_file.endswith(('.png', '.jpg', '.jpeg')):
        im_path = os.path.join(image_path, image_file)
        encoded_image = encode_image(im_path)
        image_elements.append(encoded_image)
        
# length of image elem
print(len(image_elements))
"""

### generate text, table and image summaries

In [26]:
from langchain_core.messages import HumanMessage, AIMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
from langchain.prompts import PromptTemplate

model = ChatGoogleGenerativeAI(model="gemini-pro",temperature=0, max_tokens=1024)
model_vision = ChatGoogleGenerativeAI(model="gemini-pro-vision",temperature=0, max_tokens=1024)

text & table summaries

In [ ]:
# Generate summaries of text elements
def generate_text_summaries(texts, tables, summarize_texts=False):
    """
    Summarize text elements
    texts: List of str
    tables: List of str
    summarize_texts: Bool to summarize texts
    """

    # Prompt
    prompt_text = """You are an assistant tasked with summarizing tables and text for retrieval. \
    These summaries will be embedded and used to retrieve the raw text or table elements. \
    Give a concise summary of the table or text that is well-optimized for retrieval. Table \
    or text: {element} """
    prompt = PromptTemplate.from_template(prompt_text)
    #empty_response = RunnableLambda(
      #  lambda x: AIMessage(content="Error processing document")
   # )
    # Text summary chain
    summarize_chain = {"element": lambda x: x} | prompt | model | StrOutputParser()

    # Initialize empty summaries
    text_summaries = []
    table_summaries = []

    # Apply to text if texts are provided and summarization is requested
    if texts and summarize_texts:
        text_summaries = summarize_chain.batch(texts, {"max_concurrency": 1})
    elif texts:
        text_summaries = texts

    # Apply to tables if tables are provided
    if tables:
        table_summaries = summarize_chain.batch(tables, {"max_concurrency": 1})

    return text_summaries, table_summaries


# Get text & table summaries
text_summaries, table_summaries = generate_text_summaries(texts[0:19], tables, summarize_texts=True)

In [ ]:
len(text_summaries)

In [ ]:
table_summaries

image summaries

In [29]:
import os
import base64
# encode image
def encode_image(image_path):
    """Getting the base64 string"""
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

In [31]:
def image_summarize(img_base64, prompt):
    """Make image summary"""
    msg = model_vision.invoke(
        [
            HumanMessage(
                content=[
                    {"type": "text", "text": prompt},
                    {
                        "type": "image_url",
                        "image_url": {"url": f"data:image/jpeg;base64,{img_base64}"},
                    },
                ]
            )
        ]
    )
    return msg.content


In [32]:
def generate_img_summaries(path):
    """
    Generate summaries and base64 encoded strings for images
    path: Path to list of .jpg files extracted by Unstructured
    """
    # Store base64 encoded images
    img_base64_list = []

    # Store image summaries
    image_summaries = []

    # Prompt
    prompt = """You are an assistant tasked with summarizing images for retrieval. \
    These summaries will be embedded and used to retrieve the raw image. \
    Give a concise summary of the image that is well optimized for retrieval."""

    # Apply to images
    for img_file in sorted(os.listdir(path)):
        if img_file.endswith(('.png', '.jpg', '.jpeg')):
            img_path = os.path.join(path, img_file)
            base64_image = encode_image(img_path)
            img_base64_list.append(base64_image)
            image_summaries.append(image_summarize(base64_image, prompt))

    return img_base64_list, image_summaries


In [33]:
fpath = "./figures"
# Image summaries
img_base64_list, image_summaries = generate_img_summaries(fpath)

In [ ]:
image_summaries

## Multi-vector retriever

* add raw docs and summaries to Multi-vector Retriever
* retrieve original raw documents (texts, tables & images) from *docstore* corresponding to the retrieved vector from  *vectorstore*

In [40]:
import uuid
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain.retrievers.multi_vector import MultiVectorRetriever
from langchain.schema.document import Document
from langchain.storage import InMemoryStore
from langchain.vectorstores import Chroma

In [41]:
def create_multi_vector_retriever(vectorstore, text_summaries, texts, table_summaries, tables, image_summaries, images):
    """
    Create retriever that indexes summaries, but returns raw images or texts
    """
    # Initialize the storage layer
    store = InMemoryStore()
    id_key = "doc_id"

    # Create the multi-vector retriever
    retriever = MultiVectorRetriever(
        vectorstore=vectorstore,
        docstore=store,
        id_key=id_key,
    )
    
    # Helper function to add documents to the vectorstore and docstore
    def add_documents(retriever, doc_summaries, doc_contents):
        doc_ids = [str(uuid.uuid4()) for _ in doc_contents]
        summary_docs = [
            Document(page_content=s, metadata={id_key: doc_ids[i]})
            for i, s in enumerate(doc_summaries)
        ]
        retriever.vectorstore.add_documents(summary_docs)
        retriever.docstore.mset(list(zip(doc_ids, doc_contents)))

    # Add texts, tables, and images
    # Check that text_summaries is not empty before adding
    if text_summaries:
        add_documents(retriever, text_summaries, texts)
    # Check that table_summaries is not empty before adding
    if table_summaries:
        add_documents(retriever, table_summaries, tables)
    # Check that image_summaries is not empty before adding
    if image_summaries:
        add_documents(retriever, image_summaries, images)

    return retriever



In [42]:
# The vectorstore to use to index the summaries
vectorstore = Chroma(
    collection_name="mm_rag_gemini",
    embedding_function=GoogleGenerativeAIEmbeddings(model="models/embedding-001"), # embedding model  
)

# Create retriever
retriever_multi_vector_img = create_multi_vector_retriever(
    vectorstore,
    text_summaries,
    texts,
    table_summaries,
    tables,
    image_summaries,
    img_base64_list,
)

## RAG Pipeline


In [43]:
import io
import re

from IPython.display import HTML, display
from langchain.schema.runnable import RunnableLambda, RunnablePassthrough
from PIL import Image


def plt_img_base64(img_base64):
    """Disply base64 encoded string as image"""
    # Create an HTML img tag with the base64 string as the source
    image_html = f'<img src="data:image/jpeg;base64,{img_base64}" />'
    # Display the image by rendering the HTML
    display(HTML(image_html))

def looks_like_base64(sb):
    """Check if the string looks like base64"""
    return re.match("^[A-Za-z0-9+/]+[=]{0,2}$", sb) is not None


def is_image_data(b64data):
    """
    Check if the base64 data is an image by looking at the start of the data
    """
    image_signatures = {
        b"\xFF\xD8\xFF": "jpg",
        b"\x89\x50\x4E\x47\x0D\x0A\x1A\x0A": "png",
        b"\x47\x49\x46\x38": "gif",
        b"\x52\x49\x46\x46": "webp",
    }
    try:
        header = base64.b64decode(b64data)[:8]  # Decode and get the first 8 bytes
        for sig, format in image_signatures.items():
            if header.startswith(sig):
                return True
        return False
    except Exception:
        return False

def resize_base64_image(base64_string, size=(128, 128)):
    """
    Resize an image encoded as a Base64 string
    """
    # Decode the Base64 string
    img_data = base64.b64decode(base64_string)
    img = Image.open(io.BytesIO(img_data))

    # Resize the image
    resized_img = img.resize(size, Image.LANCZOS)

    # Save the resized image to a bytes buffer
    buffered = io.BytesIO()
    resized_img.save(buffered, format=img.format)

    # Encode the resized image to Base64
    return base64.b64encode(buffered.getvalue()).decode("utf-8")

def split_image_text_types(docs):
    """
    Split base64-encoded images and texts
    """
    b64_images = []
    texts = []
    for doc in docs:
        # Check if the document is of type Document and extract page_content if so
        if isinstance(doc, Document):
            doc = doc.page_content
        if looks_like_base64(doc) and is_image_data(doc):
            doc = resize_base64_image(doc, size=(1300, 600))
            b64_images.append(doc)
        else:
            texts.append(doc)
    if len(b64_images) > 0:
        return {"images": b64_images[:1], "texts": []}
    return {"images": b64_images, "texts": texts}
  


In [44]:
def img_prompt_func(data_dict):
    """
    Join the context into a single string
    """
    formatted_texts = "\n".join(data_dict["context"]["texts"])
    messages = []

    # Adding the text for analysis
    text_message = {
        "type": "text",
        "text": (
            "You are an AI scientist tasking with providing factual answers from research papers.\n"
            "You will be given a mixed of text, tables, and image(s) usually of charts or graphs.\n"
            "Use this information to provide answers related to the user question. \n"
            f"User-provided question: {data_dict['question']}\n\n"
            "Text and / or tables:\n"
            f"{formatted_texts}"
        ),
    }
    messages.append(text_message)
    # Adding image(s) to the messages if present
    if data_dict["context"]["images"]:
        for image in data_dict["context"]["images"]:
            image_message = {
                "type": "image_url",
                "image_url": {"url": f"data:image/jpeg;base64,{image}"},
            }
            messages.append(image_message)
    return [HumanMessage(content=messages)]

def multi_modal_rag_chain(retriever):
    """
    Multi-modal RAG chain
    """

    # RAG pipeline
    chain = (
        {
            "context": retriever | RunnableLambda(split_image_text_types),
            "question": RunnablePassthrough(),
        }
        | RunnableLambda(img_prompt_func)
        | model_vision  # MM_LLM
        | StrOutputParser()
    )

    return chain

In [45]:
# Create RAG chain
chain_multimodal_rag = multi_modal_rag_chain(retriever_multi_vector_img)

examine retrieval

In [75]:
query = """How does QLORA  differ from LORA and Full-finetuned model? Explain it in detail"""
docs = retriever_multi_vector_img.get_relevant_documents(query, limit=1)

In [ ]:
len(docs)

In [ ]:
split_image_text_types(docs)

In [ ]:
docs[0]

In [ ]:
docs[1]

In [ ]:
docs[2]

In [ ]:
docs[3]

In [ ]:
# We get back relevant images
plt_img_base64(docs[2])

### Final Result

In [ ]:
chain_multimodal_rag.invoke(query)

In [81]:
query = """How is 4-bit NormalFloat better than 4-bit Float when tested on LLaMA?"""
docs = retriever_multi_vector_img.get_relevant_documents(query, limit=1)

In [ ]:
docs[1]

In [ ]:
plt_img_base64(docs[1])

In [ ]:
chain_multimodal_rag.invoke(query)